# Imports

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from uavloc.plotting.flight import plot_end_error_cdf
from uavloc.simulation.sequence import village_bootstrap

# Params

In [ ]:
RUNS = {   # (rule, replicate): run
    ("none", "A"): "034.1", ("none", "B"): "034.2", ("none", "C"): "034.3",
    ("one-class", "A"): "034.4", ("one-class", "B"): "034.5", ("one-class", "C"): "034.6",
    ("no-anchors", "A"): "034.7", ("no-anchors", "B"): "034.8", ("no-anchors", "C"): "034.9",
}
BASELINE_FLIGHTS = {"A": "../027-sequence-filters/artifacts/flights-027.12.csv",
                    "B": "../033-class-loss-in-system/artifacts/flights-033.1.csv",
                    "C": "../033-class-loss-in-system/artifacts/flights-033.4.csv"}
RULES = ["one-class", "no-anchors"]
REPLICATES = ["A", "B", "C"]
RECOVERED_M = 10.0
TAIL_M = 20.0
REPRODUCE_TOLERANCE_M = 0.01
REPRODUCE_MIN_FLIGHTS = 90
BAR_MEAN_DELTA = 0.05
BAR_MEDIAN_GUARD_M = 0.5
NAMED_FLIGHTS = [("bushcha-50p2422-25p8305", 1), ("kryve-50p1899-23p8279", 1)]
COLOURS = {"none": "#555555", "one-class": "#2a78d6", "no-anchors": "#eb6834"}
BOOTSTRAP_RESAMPLES = 10_000
SEED = 42
ARTIFACTS = Path("artifacts")
FIGURES = Path("figures")

# Data

In [ ]:
flights = {key: pd.read_csv(ARTIFACTS / f"flights-{run}.csv").assign(
               recovered=lambda t: (t["end_error_m"] <= RECOVERED_M).astype(float))
           for key, run in RUNS.items()}
{key: len(table) for key, table in flights.items()}

# Run

In [ ]:
def reproduction(replicate: str) -> dict:
    keys = ["village", "route_index"]
    ours = flights[("none", replicate)][keys + ["end_error_m"]]
    logged = pd.read_csv(BASELINE_FLIGHTS[replicate])[keys + ["end_error_m"]]
    paired = ours.merge(logged, on=keys, suffixes=("", "_logged"))
    diff = (paired["end_error_m"] - paired["end_error_m_logged"]).abs()
    matched = int((diff <= REPRODUCE_TOLERANCE_M).sum())
    return {"replicate": replicate, "flights": len(paired), "matched": matched, "max_diff_m": diff.max(),
            "passes": matched >= REPRODUCE_MIN_FLIGHTS}

In [ ]:
reproduced = pd.DataFrame([reproduction(r) for r in REPLICATES])
reproduced

In [ ]:
def system_row(table: pd.DataFrame) -> dict:
    return {"recovered_share": table["recovered"].mean(), "median_end_error_m": table["end_error_m"].median(),
            "mean_end_error_m": table["end_error_m"].mean(), "divergence_share": table["diverged"].mean(),
            "skipped_share": table["skipped_share"].mean(), "flights_over_20m": int((table["end_error_m"] > TAIL_M).sum())}

In [ ]:
system = pd.DataFrame({key: system_row(table) for key, table in flights.items()}).T
system.index.names = ["rule", "replicate"]
system.round(3)

In [ ]:
def paired_row(rule: str, replicate: str) -> dict:
    a, b = flights[(rule, replicate)], flights[("none", replicate)]
    rec = village_bootstrap(a, b, "recovered", BOOTSTRAP_RESAMPLES, SEED)
    end = village_bootstrap(a, b, "end_error_m", BOOTSTRAP_RESAMPLES, SEED)
    return {"rule": rule, "replicate": replicate,
            "delta_recovered": rec[0], "recovered_ci_low": rec[1], "recovered_ci_high": rec[2],
            "delta_mean_end_m": end[0], "mean_end_ci_low": end[1], "mean_end_ci_high": end[2],
            "delta_median_m": system.loc[(rule, replicate), "median_end_error_m"]
                              - system.loc[("none", replicate), "median_end_error_m"]}

In [ ]:
paired = pd.DataFrame([paired_row(rule, r) for rule in RULES for r in REPLICATES])
paired.round(3)

In [ ]:
def bar_verdict(rule: str) -> dict:
    rows = paired[paired["rule"] == rule]
    checks = {"1_mean_delta": rows["delta_recovered"].mean() >= BAR_MEAN_DELTA,
              "2_every_replicate": bool((rows["delta_recovered"] >= 0).all()),
              "3_median_guard": bool((rows["delta_median_m"] <= BAR_MEDIAN_GUARD_M).all())}
    return {"rule": rule, "mean_delta_recovered": rows["delta_recovered"].mean(), **checks,
            "passes": all(checks.values())}

In [ ]:
verdicts = pd.DataFrame([bar_verdict(rule) for rule in RULES])
verdicts

In [ ]:
named = pd.DataFrame({f"{rule} {r}": table.set_index(["village", "route_index"]).loc[NAMED_FLIGHTS, "end_error_m"]
                      for (rule, r), table in flights.items()}).T
named.round(1)

In [ ]:
results = {"reproduction": reproduced.to_dict("records"), "system": system.reset_index().to_dict("records"),
           "paired": paired.to_dict("records"), "verdicts": verdicts.to_dict("records")}
(ARTIFACTS / "analysis.json").write_text(json.dumps(results, indent=1, default=float))

# Plots

In [ ]:
FIGURES.mkdir(exist_ok=True)
fig, axes = plt.subplots(1, len(REPLICATES), figsize=(15, 4.2), sharey=True)
for ax, replicate in zip(axes, REPLICATES):
    plot_end_error_cdf({rule: flights[(rule, replicate)]["end_error_m"] for rule in ["none"] + RULES}, ax)
    ax.axvline(RECOVERED_M, color="black", lw=0.8, ls="--")
    ax.set_title(f"replicate {replicate}")
fig.suptitle("034: end error on the 94 test flights, by blind rule (dashed: 10 m)")
fig.tight_layout()
fig.savefig(FIGURES / "end-error-cdf.png", dpi=150)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.2))
for rule in ["none"] + RULES:
    rows = system.loc[rule]
    ax.scatter(rows["skipped_share"], rows["recovered_share"], color=COLOURS[rule], s=50, label=rule)
    for replicate, row in rows.iterrows():
        ax.annotate(replicate, (row["skipped_share"], row["recovered_share"]), xytext=(4, 3), textcoords="offset points")
ax.set(xlabel="share of fixes skipped as blind", ylabel=f"share of flights ending ≤ {RECOVERED_M:g} m")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "skipped-vs-recovered.png", dpi=150)